# NextGenUp — Google Colab

Run NextGenUp (AI image/video studio) on Google Colab with a public ngrok URL.

Run cells in order.

In [ ]:
!nvidia-smi 2>/dev/null || echo 'No GPU (CPU mode is fine)'

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null 2>&1
!ffmpeg -version | head -1

In [ ]:
import os, shutil
REPO_URL = "https://github.com/riponcm/nextgenUp.git"
DEST = "/content/nextgenUp"
if os.path.exists(DEST):
    shutil.rmtree(DEST)
!git clone "$REPO_URL" "$DEST"
print("Cloned. Files:", os.listdir(DEST))

In [ ]:
!pip install -q -r /content/nextgenUp/requirements.txt
print("Dependencies installed")

In [ ]:
import os
os.makedirs("/content/nextgenUp/static/models", exist_ok=True)
model = "/content/nextgenUp/static/models/realesr-general-x4v3.onnx"
if not os.path.exists(model):
    print("Downloading Real-ESRGAN ONNX model (~5MB)...")
    import urllib.request
    urllib.request.urlretrieve(
        "https://huggingface.co/OwlMaster/AllFilesRope/resolve/main/realesr-general-x4v3.onnx",
        model
    )
    print("Model downloaded.")
else:
    print("Model already exists.")

In [ ]:
from google.colab import userdata
import os
os.environ['NGROK_TOKEN'] = userdata.get('NGROK_TOKEN')
print("NGROK_TOKEN loaded")

In [ ]:
import os, threading, time
from pyngrok import ngrok

token = os.environ.get('NGROK_TOKEN', '')
if not token:
    print('ERROR: NGROK_TOKEN not set')
else:
    import os as _os
    ngrok_cfg = _os.path.expanduser('~/.config/ngrok/ngrok.yml')
    if _os.path.exists(ngrok_cfg):
        _os.remove(ngrok_cfg)
    ngrok.set_auth_token(token)
    os.chdir('/content/nextgenUp')
    import subprocess
    proc = subprocess.Popen(
        ['python', 'app.py'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    time.sleep(5)
    public_url = ngrok.connect(5000, 'http').public_url
    print('=' * 60)
    print(f'NextGenUp URL: {public_url}')
    print('=' * 60)
    print('Open the URL above in your browser.')
    print('Keep this cell running.')
    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        print('Shutting down...')
        ngrok.kill()
        proc.terminate()
